In [2]:
pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [3]:
import yfinance as yf
import pandas as pd

stocks = ["TCS.NS", "INFY.NS", "RELIANCE.NS"]

all_data = []

for stock in stocks:

    df = yf.download(
        stock,
        start="2015-01-01",
        end="2025-01-01"
    )
    
    # Flatten multi-level columns returned by newer yfinance versions
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # reset date from index into column
    df.reset_index(inplace=True)

    # add company column
    df["Company"] = stock

    # keep only needed columns
    df = df[[
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "Company"
    ]]

    all_data.append(df)

# vertical concatenation
fdf = pd.concat(all_data, ignore_index=True)

print(fdf.head())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Price       Date        Open        High         Low       Close   Volume  \
0     2015-01-01  983.872030  983.872030  973.906828  975.650757   366830   
1     2015-01-02  977.739708  993.051607  977.586435  988.643921   925740   
2     2015-01-05  989.237984  996.481883  967.640287  973.619446  1754242   
3     2015-01-06  969.345877  969.345877  935.195859  937.725525  2423784   
4     2015-01-07  946.694335  950.201282  922.720336  926.648926  2636332   

Price Company  
0      TCS.NS  
1      TCS.NS  
2      TCS.NS  
3      TCS.NS  
4      TCS.NS  


In [4]:
fdf.head()


Price,Date,Open,High,Low,Close,Volume,Company
0,2015-01-01,983.872030,983.872030,973.906828,975.650757,366830,TCS.NS
1,2015-01-02,977.739708,993.051607,977.586435,988.643921,925740,TCS.NS
2,2015-01-05,989.237984,996.481883,967.640287,973.619446,1754242,TCS.NS
3,2015-01-06,969.345877,969.345877,935.195859,937.725525,2423784,TCS.NS
4,2015-01-07,946.694335,950.201282,922.720336,926.648926,2636332,TCS.NS


In [5]:
fdf.describe()

Price,Date,Open,High,Low,Close,Volume
count,7401,7401.000000,7401.000000,7401.000000,7401.000000,7.401000e+03
mean,2020-01-01 11:22:56.084312832,1235.696323,1247.778702,1223.056852,1235.237103,9.704301e+06
min,2015-01-01 00:00:00,174.742467,174.977847,170.430607,173.490631,8.682200e+04
25%,2017-07-04 00:00:00,521.302830,529.351649,510.893989,519.613098,2.963643e+06
50%,2020-01-07 00:00:00,1016.100269,1027.443572,1004.774469,1015.317444,6.621839e+06
75%,2022-06-30 00:00:00,1534.939208,1554.826428,1518.469280,1537.341797,1.241994e+07
max,2024-12-31 00:00:00,4274.691352,4289.871200,4214.905459,4253.906250,1.644050e+08
std,NaN,916.564963,924.667438,908.360131,916.421537,1.099946e+07


In [6]:
fdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7401 entries, 0 to 7400
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Date     7401 non-null   datetime64[ns]
 1   Open     7401 non-null   float64       
 2   High     7401 non-null   float64       
 3   Low      7401 non-null   float64       
 4   Close    7401 non-null   float64       
 5   Volume   7401 non-null   int64         
 6   Company  7401 non-null   object        
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 404.9+ KB


In [7]:
fdf.isnull().sum()

Price
Date       0
Open       0
High       0
Low        0
Close      0
Volume     0
Company    0
dtype: int64

## Feature Engg

Adding column RETURN

In [8]:
fdf["Return"] = fdf.groupby("Company")[
    "Close"
].pct_change()

Adding column MOVING AVGS

In [9]:
fdf["MA10"] = fdf.groupby("Company")[
    "Close"
].transform(lambda x: x.rolling(10).mean())

fdf["MA50"] = fdf.groupby("Company")[
    "Close"
].transform(lambda x: x.rolling(50).mean())

adding column VOLATILITY

In [10]:
fdf["Volatility"] = fdf.groupby(
    "Company"
)["Return"].transform(
    lambda x: x.rolling(10).std()
)

adding column DAILY MOMENTUM

In [11]:
fdf["Momentum"] = fdf.groupby(
    "Company"
)["Close"].diff(5)

adding Column RSI(>70 overbought & <30 oversold)

In [12]:
delta = fdf.groupby("Company")[
    "Close"
].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

fdf["RSI"] = 100 - (100 / (1 + rs))

fdf["Range"] = (
    fdf["High"] - fdf["Low"]
)

fdf["MA_Ratio"] = (
    fdf["MA10"] /
    fdf["MA50"]
)




In [13]:
fdf.describe()

Price,Date,Open,High,Low,Close,Volume,Return,MA10,MA50,Volatility,Momentum,RSI,Range,MA_Ratio
count,7401,7401.000000,7401.000000,7401.000000,7401.000000,7.401000e+03,7398.000000,7374.000000,7254.000000,7371.000000,7386.000000,7359.000000,7401.000000,7254.000000
mean,2020-01-01 11:22:56.084312832,1235.696323,1247.778702,1223.056852,1235.237103,9.704301e+06,0.000786,1234.583543,1231.048375,0.014775,3.679881,53.245907,24.721850,1.013677
min,2015-01-01 00:00:00,174.742467,174.977847,170.430607,173.490631,8.682200e+04,-0.161881,177.740446,185.221892,0.003087,-357.598389,7.280590,1.435262,0.742670
25%,2017-07-04 00:00:00,521.302830,529.351649,510.893989,519.613098,2.963643e+06,-0.007898,524.140477,536.407027,0.010518,-13.995674,40.898197,10.016894,0.983141
50%,2020-01-07 00:00:00,1016.100269,1027.443572,1004.774469,1015.317444,6.621839e+06,0.000576,1014.351727,1002.972895,0.013338,2.667145,53.565734,18.177702,1.014496
75%,2022-06-30 00:00:00,1534.939208,1554.826428,1518.469280,1537.341797,1.241994e+07,0.009396,1539.574356,1537.289272,0.017102,22.083557,65.545151,32.219699,1.043169
max,2024-12-31 00:00:00,4274.691352,4289.871200,4214.905459,4253.906250,1.644050e+08,0.147180,4216.115283,4094.188481,0.090672,378.760742,97.543831,267.273149,1.215321
std,NaN,916.564963,924.667438,908.360131,916.421537,1.099946e+07,0.016380,914.530312,905.455215,0.007206,49.304557,16.533030,21.749139,0.049695


In [14]:
fdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7401 entries, 0 to 7400
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        7401 non-null   datetime64[ns]
 1   Open        7401 non-null   float64       
 2   High        7401 non-null   float64       
 3   Low         7401 non-null   float64       
 4   Close       7401 non-null   float64       
 5   Volume      7401 non-null   int64         
 6   Company     7401 non-null   object        
 7   Return      7398 non-null   float64       
 8   MA10        7374 non-null   float64       
 9   MA50        7254 non-null   float64       
 10  Volatility  7371 non-null   float64       
 11  Momentum    7386 non-null   float64       
 12  RSI         7359 non-null   float64       
 13  Range       7401 non-null   float64       
 14  MA_Ratio    7254 non-null   float64       
dtypes: datetime64[ns](1), float64(12), int64(1), object(1)
memory usage: 867

In [15]:
fdf=fdf.dropna()
fdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7254 entries, 49 to 7400
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        7254 non-null   datetime64[ns]
 1   Open        7254 non-null   float64       
 2   High        7254 non-null   float64       
 3   Low         7254 non-null   float64       
 4   Close       7254 non-null   float64       
 5   Volume      7254 non-null   int64         
 6   Company     7254 non-null   object        
 7   Return      7254 non-null   float64       
 8   MA10        7254 non-null   float64       
 9   MA50        7254 non-null   float64       
 10  Volatility  7254 non-null   float64       
 11  Momentum    7254 non-null   float64       
 12  RSI         7254 non-null   float64       
 13  Range       7254 non-null   float64       
 14  MA_Ratio    7254 non-null   float64       
dtypes: datetime64[ns](1), float64(12), int64(1), object(1)
memory usage: 906.8+ 

##### extracting date and month (donno why but still doing it)

In [16]:
fdf["Month"] = (
    fdf["Date"].dt.month
)

fdf["DayOfWeek"] = (
    fdf["Date"].dt.dayofweek
)

### creating target variable 
if close after a week(5 trading days) > today's close its 1 else 0.

In [38]:
fdf["Target"] = (
    fdf.groupby("Company")["Close"]
    .shift(-5) > fdf["Close"]
)

fdf = fdf.dropna(subset=["Target"])

fdf["Target"] = fdf["Target"].astype(int)

In [39]:
fdf.head()

Price,Date,Open,High,Low,Close,Volume,Company,Return,MA10,MA50,Volatility,Momentum,RSI,Range,MA_Ratio,Month,DayOfWeek,Target,Company_Code
49,2015-03-16,992.830212,1002.354519,980.079974,984.035645,1556296,TCS.NS,-0.007765,1019.544672,985.140916,0.017373,-32.452148,33.799654,22.274545,1.034923,3,0,1,2
50,2015-03-17,991.793439,1001.087283,982.288351,992.254272,1267318,TCS.NS,0.008352,1016.253375,985.472986,0.017694,-22.658813,35.857129,18.798931,1.031234,3,1,1,2
51,2015-03-18,995.038502,998.302875,979.964779,982.902771,1255170,TCS.NS,-0.009425,1007.933020,985.358163,0.009192,-18.126831,36.557620,18.338096,1.022910,3,2,1,2
52,2015-03-19,989.297207,1000.760956,986.263297,997.074097,2448964,TCS.NS,0.014418,1002.181909,985.827256,0.011519,-8.756348,43.173202,14.497659,1.016590,3,3,0,2
53,2015-03-20,996.978348,1005.427317,990.392030,1002.719788,1963908,TCS.NS,0.005662,998.898370,987.127141,0.011089,10.983582,44.285110,15.035287,1.011925,3,4,0,2


#### encoding the companies

In [40]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

fdf["Company_Code"] = (
    encoder.fit_transform(
        fdf["Company"]
    )
)

In [41]:
fdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7254 entries, 49 to 7400
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          7254 non-null   datetime64[ns]
 1   Open          7254 non-null   float64       
 2   High          7254 non-null   float64       
 3   Low           7254 non-null   float64       
 4   Close         7254 non-null   float64       
 5   Volume        7254 non-null   int64         
 6   Company       7254 non-null   object        
 7   Return        7254 non-null   float64       
 8   MA10          7254 non-null   float64       
 9   MA50          7254 non-null   float64       
 10  Volatility    7254 non-null   float64       
 11  Momentum      7254 non-null   float64       
 12  RSI           7254 non-null   float64       
 13  Range         7254 non-null   float64       
 14  MA_Ratio      7254 non-null   float64       
 15  Month         7254 non-null   int32       

### adding nlp sentiment part 
using marketaux api for news as marketaux provides comprehensive market data, including historical company news ,Also it can handle requests in large volume. and finbert for sentiment analysis as its specifically trained on finanial words so it can more accurately sort bullish and bearish.


this is gonna be a full nlp pipeline . from cleaning to tokenization to final sentiment that will be used as a feature in the above dataset.

In [42]:
ndf=pd.read_csv(r"C:\Users\arije\Downloads\archive (5)\stock_news_2016 to 2026.csv")
ndf.head()

,date,title,description,url,source_file,categories,matched_keywords,relevance_score,has_negation,impact_tier
0,2016-01-01,Lending to foreign step-down arms of Indian fi...,RBI slaps 2% additional provision for such loans,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,rbi,2,False,LOW
1,2016-01-02,IDBI Bank to raise Rs 900 cr through Basel-III...,Many banks have been raising money through tie...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,basel,2,False,LOW
2,2016-01-02,Forex reserves up $943 mn,India's foreign exchange reserves rose $943 mi...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,macro_government,forex reserve,3,False,MEDIUM
3,2016-01-02,SBI: Lending rate cut unlikely till end-Mar,Country's largest lender SBI today ruled out f...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,stock_specific,sbi,2,True,LOW
4,2016-01-03,"ASK Group to invest Rs 1,500 cr in real estate...",ASK Group plans to step up its equity investme...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_cement_infra,real estate,1,False,LOW


In [43]:
ndf.drop(columns=["url","source_file","categories"], inplace=True)

In [44]:
df.describe()   

Price,Date,Open,High,Low,Close,Volume
count,2467,2467.000000,2467.000000,2467.000000,2467.000000,2.467000e+03
mean,2020-01-01 11:22:56.084313088,731.223434,738.686446,723.345324,730.697570,1.842886e+07
min,2015-01-01 00:00:00,174.742467,174.977847,170.430607,173.490631,1.705656e+06
25%,2017-07-04 12:00:00,315.861174,317.608673,313.160472,315.653000,1.069616e+07
50%,2020-01-07 00:00:00,645.396295,652.978879,636.583512,642.041077,1.464381e+07
75%,2022-06-29 12:00:00,1122.798986,1133.629472,1112.048646,1121.365723,2.109197e+07
max,2024-12-31 00:00:00,1592.662006,1596.980166,1573.851280,1589.138184,1.426834e+08
std,NaN,420.153244,423.754485,416.275052,419.850320,1.355224e+07


In [45]:
keywords = [
    "TCS",
    "Tata Consultancy",
    "Infosys",
    "Reliance"
]

pattern = "|".join(keywords)

In [46]:
filtered_news = ndf[
    ndf["title"]
    .str.contains(
        pattern,
        case=False,
        na=False
    )
]

In [47]:
print(filtered_news.shape)

print(filtered_news.head())

(6397, 7)
            date                                              title  \
146   2016-02-11  CCI approves Nippon Life stake hike in Relianc...   
978   2016-09-07  Reliance Capital raises Rs 2,000 cr though deb...   
979   2016-09-07  Reliance Capital raises ~2,000 cr though deben...   
1000  2016-09-13  Reliance Capital to list home finance biz; eye...   
1032  2016-09-21          Reliance Nippon Life bullish on UP market   

                                            description matched_keywords  \
146   Nippon Life had announced its plan to hike the...         reliance   
978   Funds will be utilised for refinancing existin...         reliance   
979   The finance company will use the proceeds to r...         reliance   
1000  All shareholders of Reliance Capital will rece...         reliance   
1032  Insurer to ramp up advisors' network to 25,000...         reliance   

      relevance_score  has_negation impact_tier  
146                 2         False         LOW  
978   

In [48]:
def assign_company(headline):

    headline = headline.lower()

    if "tcs" in headline or \
       "tata consultancy" in headline:

        return "TCS.NS"

    elif "infosys" in headline:

        return "INFY.NS"

    elif "reliance" in headline:

        return "RELIANCE.NS"

    else:
        return None

In [49]:
filtered_news["Company"] = (
    filtered_news["title"]
    .apply(assign_company)
)

C:\Users\arije\AppData\Local\Temp\ipykernel_7996\2828545426.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_news["Company"] = (


In [50]:
filtered_news.dropna(
    subset=["Company"],
    inplace=True
)

C:\Users\arije\AppData\Local\Temp\ipykernel_7996\2600320720.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_news.dropna(


In [51]:
print(
    filtered_news["Company"]
    .value_counts()
)

Company
RELIANCE.NS    2968
TCS.NS         1847
INFY.NS        1582
Name: count, dtype: int64


In [52]:
filtered_news.head()    

,date,title,description,matched_keywords,relevance_score,has_negation,impact_tier,Company
146,2016-02-11,CCI approves Nippon Life stake hike in Relianc...,Nippon Life had announced its plan to hike the...,reliance,2,False,LOW,RELIANCE.NS
978,2016-09-07,"Reliance Capital raises Rs 2,000 cr though deb...",Funds will be utilised for refinancing existin...,reliance,2,False,LOW,RELIANCE.NS
979,2016-09-07,"Reliance Capital raises ~2,000 cr though deben...",The finance company will use the proceeds to r...,reliance,2,False,LOW,RELIANCE.NS
1000,2016-09-13,Reliance Capital to list home finance biz; eye...,All shareholders of Reliance Capital will rece...,reliance,2,False,LOW,RELIANCE.NS
1032,2016-09-21,Reliance Nippon Life bullish on UP market,"Insurer to ramp up advisors' network to 25,000...",reliance,2,False,LOW,RELIANCE.NS


since the dataset is ready now we use FinBERT.


In [53]:
from transformers import pipeline
classifier = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [54]:
headline = filtered_news.iloc[0]["title"]

print(headline)

CCI approves Nippon Life stake hike in Reliance Life to 49%


In [55]:
result = classifier(headline)

print(result)

[{'label': 'positive', 'score': 0.8691171407699585}]


In [56]:
def get_sentiment_score(text):

    result = classifier(text)[0]

    label = result["label"]
    score = result["score"]

    if label == "positive":
        return score

    elif label == "negative":
        return -score

    else:
        return 0

In [57]:
sample_news = filtered_news.head(20).copy()
sample_news["Sentiment"] = (
    sample_news["title"]
    .apply(get_sentiment_score)

)
print(
    sample_news[
        ["title", "Sentiment"]
    ]
)

                                                  title  Sentiment
146   CCI approves Nippon Life stake hike in Relianc...   0.869117
978   Reliance Capital raises Rs 2,000 cr though deb...   0.000000
979   Reliance Capital raises ~2,000 cr though deben...   0.000000
1000  Reliance Capital to list home finance biz; eye...   0.000000
1032          Reliance Nippon Life bullish on UP market   0.427561
1036  Andhra Bank ties up with Cigna TTK, Reliance G...   0.000000
1199  Reliance Home Finance plans to garner Rs 3,500...   0.576012
1309  Opening bell : Asia opens mixed ; SBI , Infosy...   0.686420
1340  Sensex , Nifty likely to be flat ; Infosys , a...   0.561786
1413  Sensex , Nifty open flat ; Tata Motors , Infos...   0.894817
1424  Lack of investments in new tech may crimp TCS ...  -0.943057
1467  Sensex rises over 200 pts , Nifty strong ; TCS...   0.931673
1542  Sensex , Nifty fall for 2nd day ; TCS rebounds...  -0.685387
1584  Nifty above 8250 , Sensex climbs over 100 pts ...   0.90

In [66]:

filtered_news["Sentiment"] = (
    filtered_news["title"]
    .apply(get_sentiment_score)
)

C:\Users\arije\AppData\Local\Temp\ipykernel_7996\2793774136.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_news["Sentiment"] = (


In [67]:
filtered_news.to_csv(
    "news_with_sentiment.csv",
    index=False
)

In [68]:
sentiment_df = pd.read_csv(
    "news_with_sentiment.csv"
)
sentiment_df = sentiment_df[
    ["date", "Company", "Sentiment"]
]

In [69]:
sentiment_df.head() 

,date,Company,Sentiment
0,2016-02-11,RELIANCE.NS,0.869117
1,2016-09-07,RELIANCE.NS,0.000000
2,2016-09-07,RELIANCE.NS,0.000000
3,2016-09-13,RELIANCE.NS,0.000000
4,2016-09-21,RELIANCE.NS,0.427561


In [70]:
fdf.rename(columns={"Date": "date"}, inplace=True)

In [71]:
fdf.head()

Price,date,Open,High,Low,Close,Volume,Company,Return,MA10,MA50,Volatility,Momentum,RSI,Range,MA_Ratio,Month,DayOfWeek,Target,Company_Code
49,2015-03-16,992.830212,1002.354519,980.079974,984.035645,1556296,TCS.NS,-0.007765,1019.544672,985.140916,0.017373,-32.452148,33.799654,22.274545,1.034923,3,0,1,2
50,2015-03-17,991.793439,1001.087283,982.288351,992.254272,1267318,TCS.NS,0.008352,1016.253375,985.472986,0.017694,-22.658813,35.857129,18.798931,1.031234,3,1,1,2
51,2015-03-18,995.038502,998.302875,979.964779,982.902771,1255170,TCS.NS,-0.009425,1007.933020,985.358163,0.009192,-18.126831,36.557620,18.338096,1.022910,3,2,1,2
52,2015-03-19,989.297207,1000.760956,986.263297,997.074097,2448964,TCS.NS,0.014418,1002.181909,985.827256,0.011519,-8.756348,43.173202,14.497659,1.016590,3,3,0,2
53,2015-03-20,996.978348,1005.427317,990.392030,1002.719788,1963908,TCS.NS,0.005662,998.898370,987.127141,0.011089,10.983582,44.285110,15.035287,1.011925,3,4,0,2


### now merging the two datasets

problem occured with name.(so renamed Date as date). then date column was of different datatypes in the two datasets. one of object and other of date-time

In [63]:
# fdf = pd.merge(
#     fdf,
#     sentiment_df 
# ,
#     on=["date", "Company"],
#     how="left"
# )

In [64]:
fdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7254 entries, 49 to 7400
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          7254 non-null   datetime64[ns]
 1   Open          7254 non-null   float64       
 2   High          7254 non-null   float64       
 3   Low           7254 non-null   float64       
 4   Close         7254 non-null   float64       
 5   Volume        7254 non-null   int64         
 6   Company       7254 non-null   object        
 7   Return        7254 non-null   float64       
 8   MA10          7254 non-null   float64       
 9   MA50          7254 non-null   float64       
 10  Volatility    7254 non-null   float64       
 11  Momentum      7254 non-null   float64       
 12  RSI           7254 non-null   float64       
 13  Range         7254 non-null   float64       
 14  MA_Ratio      7254 non-null   float64       
 15  Month         7254 non-null   int32       

made a mistake of not creating the aggregate datset so the overlapping vlaues in the join were too less.

In [72]:
daily_sentiment = (
    sentiment_df.groupby(
        ["date", "Company"]
    )["Sentiment"]
    .mean()
    .reset_index()
)

In [73]:
daily_sentiment.head()

,date,Company,Sentiment
0,2016-02-11,RELIANCE.NS,0.869117
1,2016-09-07,RELIANCE.NS,0.000000
2,2016-09-13,RELIANCE.NS,0.000000
3,2016-09-21,RELIANCE.NS,0.427561
4,2016-09-23,RELIANCE.NS,0.000000


In [74]:
daily_sentiment["date"] = pd.to_datetime(
    daily_sentiment["date"])

In [75]:
print(fdf["date"].dtype)

print(daily_sentiment["date"].dtype)

datetime64[ns]
datetime64[ns]


In [76]:
fdf = pd.merge(
    fdf,
    daily_sentiment 
,
    on=["date", "Company"],
    how="left"
)

In [77]:
fdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7254 entries, 0 to 7253
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          7254 non-null   datetime64[ns]
 1   Open          7254 non-null   float64       
 2   High          7254 non-null   float64       
 3   Low           7254 non-null   float64       
 4   Close         7254 non-null   float64       
 5   Volume        7254 non-null   int64         
 6   Company       7254 non-null   object        
 7   Return        7254 non-null   float64       
 8   MA10          7254 non-null   float64       
 9   MA50          7254 non-null   float64       
 10  Volatility    7254 non-null   float64       
 11  Momentum      7254 non-null   float64       
 12  RSI           7254 non-null   float64       
 13  Range         7254 non-null   float64       
 14  MA_Ratio      7254 non-null   float64       
 15  Month         7254 non-null   int32   

In [78]:
fdf["Sentiment"] = (fdf["Sentiment"].fillna(0))

Dont know much about finance but this advice from gpt seemed fair " to shift teh sentiment scores by one day to avoid temporal leakage"


In [79]:
fdf["Sentiment"] = (fdf.groupby("Company")["Sentiment"].shift(1))
fdf.dropna(inplace=True)

In [83]:
fdf.head()

,date,Open,High,Low,Close,Volume,Company,Return,MA10,MA50,Volatility,Momentum,RSI,Range,MA_Ratio,Month,DayOfWeek,Target,Company_Code,Sentiment
1,2015-03-17,991.793439,1001.087283,982.288351,992.254272,1267318,TCS.NS,0.008352,1016.253375,985.472986,0.017694,-22.658813,35.857129,18.798931,1.031234,3,1,1,2,0.0
2,2015-03-18,995.038502,998.302875,979.964779,982.902771,1255170,TCS.NS,-0.009425,1007.933020,985.358163,0.009192,-18.126831,36.557620,18.338096,1.022910,3,2,1,2,0.0
3,2015-03-19,989.297207,1000.760956,986.263297,997.074097,2448964,TCS.NS,0.014418,1002.181909,985.827256,0.011519,-8.756348,43.173202,14.497659,1.016590,3,3,0,2,0.0
4,2015-03-20,996.978348,1005.427317,990.392030,1002.719788,1963908,TCS.NS,0.005662,998.898370,987.127141,0.011089,10.983582,44.285110,15.035287,1.011925,3,4,0,2,0.0
5,2015-03-23,1004.658912,1010.419571,1000.818473,1005.100525,1566608,TCS.NS,0.002374,997.759644,988.696173,0.009784,21.064880,44.179490,9.601098,1.009167,3,0,0,2,0.0


In [85]:
y = fdf["Target"]

In [86]:
features = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Return",
    "MA10",
    "MA50",
    "Volatility",
    "Momentum",
    "RSI",
    "Range",
    "MA_Ratio",
    "Month",
    "DayOfWeek",
    "Company_Code",
    "Sentiment"
]

sorting the data

In [87]:
fdf = fdf.sort_values(
    ["Company", "date"]
).reset_index(drop=True)

training one ompany at a time to check results for different models

In [88]:
company_df = fdf[
    fdf["Company"] == "TCS.NS"
].copy()

In [89]:
X = company_df[features]

y = company_df["Target"]

In [90]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(
    n_splits=5
)

In [92]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

rf_acc = []
rf_f1 = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        random_state=42
    )

    rf.fit(X_train, y_train)

    pred = rf.predict(X_test)

    rf_acc.append(
        accuracy_score(y_test, pred)
    )

    rf_f1.append(
        f1_score(y_test, pred)
    )

    print(
        f"Fold {fold+1}: "
        f"Acc={rf_acc[-1]:.4f}, "
        f"F1={rf_f1[-1]:.4f}"
    )

print("\nMean Accuracy:",
      sum(rf_acc)/len(rf_acc))

print("Mean F1:",
      sum(rf_f1)/len(rf_f1))

Fold 1: Acc=0.4826, F1=0.4348
Fold 2: Acc=0.5199, F1=0.6365
Fold 3: Acc=0.4229, F1=0.2215
Fold 4: Acc=0.5771, F1=0.7059
Fold 5: Acc=0.5373, F1=0.3212

Mean Accuracy: 0.5079601990049751
Mean F1: 0.4639688389680584


In [93]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score

xgb_acc = []
xgb_f1 = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    xgb.fit(X_train, y_train)

    pred = xgb.predict(X_test)

    xgb_acc.append(
        accuracy_score(y_test, pred)
    )

    xgb_f1.append(
        f1_score(y_test, pred)
    )

    print(
        f"Fold {fold+1}: "
        f"Acc={xgb_acc[-1]:.4f}, "
        f"F1={xgb_f1[-1]:.4f}"
    )

print("\nMean Accuracy:",
      sum(xgb_acc)/len(xgb_acc))

print("Mean F1:",
      sum(xgb_f1)/len(xgb_f1))

Fold 1: Acc=0.4925, F1=0.4205
Fold 2: Acc=0.5149, F1=0.6435
Fold 3: Acc=0.4527, F1=0.2949
Fold 4: Acc=0.5796, F1=0.6933
Fold 5: Acc=0.5100, F1=0.2213

Mean Accuracy: 0.5099502487562189
Mean F1: 0.4546930410335725


In [94]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

svm_acc = []
svm_f1 = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    )

    X_test_scaled = scaler.transform(
        X_test
    )

    svm = SVC(
        probability=True,
        kernel="rbf",
        C=1,
        random_state=42
    )

    svm.fit(
        X_train_scaled,
        y_train
    )

    pred = svm.predict(
        X_test_scaled
    )

    svm_acc.append(
        accuracy_score(y_test, pred)
    )

    svm_f1.append(
        f1_score(y_test, pred)
    )

    print(
        f"Fold {fold+1}: "
        f"Acc={svm_acc[-1]:.4f}, "
        f"F1={svm_f1[-1]:.4f}"
    )

print("\nMean Accuracy:",
      sum(svm_acc)/len(svm_acc))

print("Mean F1:",
      sum(svm_f1)/len(svm_f1))

Fold 1: Acc=0.5224, F1=0.5733
Fold 2: Acc=0.4975, F1=0.6431
Fold 3: Acc=0.4080, F1=0.1377
Fold 4: Acc=0.6144, F1=0.7070
Fold 5: Acc=0.5299, F1=0.4075

Mean Accuracy: 0.5144278606965174
Mean F1: 0.49372837465678954


In [95]:
print("RF  Accuracy:", sum(rf_acc)/len(rf_acc))
print("RF  F1      :", sum(rf_f1)/len(rf_f1))

print("XGB Accuracy:", sum(xgb_acc)/len(xgb_acc))
print("XGB F1      :", sum(xgb_f1)/len(xgb_f1))

print("SVM Accuracy:", sum(svm_acc)/len(svm_acc))
print("SVM F1      :", sum(svm_f1)/len(svm_f1))

RF  Accuracy: 0.5079601990049751
RF  F1      : 0.4639688389680584
XGB Accuracy: 0.5099502487562189
XGB F1      : 0.4546930410335725
SVM Accuracy: 0.5144278606965174
SVM F1      : 0.49372837465678954


In [96]:
print(y.value_counts(normalize=True))

Target
1    0.541994
0    0.458006
Name: proportion, dtype: float64


## sentiment feature might be hurting the model as noise instead of helping

In [97]:
features_relative = [
    "Return",
    "Volatility",
    "Momentum",
    "RSI",
    "MA_Ratio",
    "Range",
    "Company_Code"
]

In [98]:
X = fdf[features_relative]

y = fdf["Target"]

In [99]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

rf_acc = []
rf_f1 = []

for train_idx, test_idx in tscv.split(X):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        random_state=42
    )

    rf.fit(X_train, y_train)

    pred = rf.predict(X_test)

    rf_acc.append(
        accuracy_score(y_test, pred)
    )

    rf_f1.append(
        f1_score(y_test, pred)
    )

print("RF Accuracy:", sum(rf_acc)/len(rf_acc))
print("RF F1      :", sum(rf_f1)/len(rf_f1))

RF Accuracy: 0.5350993377483444
RF F1      : 0.6392552746163019


In [101]:
from xgboost import XGBClassifier

xgb_acc = []
xgb_f1 = []

for train_idx, test_idx in tscv.split(X):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    xgb.fit(X_train, y_train)

    pred = xgb.predict(X_test)

    xgb_acc.append(
        accuracy_score(y_test, pred)
    )

    xgb_f1.append(
        f1_score(y_test, pred)
    )

print("XGB Accuracy:", sum(xgb_acc)/len(xgb_acc))
print("XGB F1      :", sum(xgb_f1)/len(xgb_f1))

XGB Accuracy: 0.5127483443708609
XGB F1      : 0.5746595957725993


In [102]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

svm_acc = []
svm_f1 = []

for train_idx, test_idx in tscv.split(X):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    )

    X_test_scaled = scaler.transform(
        X_test
    )

    svm = SVC(
        probability=True,
        kernel="rbf",
        C=1,
        random_state=42
    )

    svm.fit(
        X_train_scaled,
        y_train
    )

    pred = svm.predict(
        X_test_scaled
    )

    svm_acc.append(
        accuracy_score(y_test, pred)
    )

    svm_f1.append(
        f1_score(y_test, pred)
    )

print("SVM Accuracy:", sum(svm_acc)/len(svm_acc))
print("SVM F1      :", sum(svm_f1)/len(svm_f1))

SVM Accuracy: 0.5172185430463576
SVM F1      : 0.5584778799014398


In [103]:
print("="*40)

print("RF Accuracy :", sum(rf_acc)/len(rf_acc))
print("RF F1       :", sum(rf_f1)/len(rf_f1))

print()

print("XGB Accuracy:", sum(xgb_acc)/len(xgb_acc))
print("XGB F1      :", sum(xgb_f1)/len(xgb_f1))

print()

print("SVM Accuracy:", sum(svm_acc)/len(svm_acc))
print("SVM F1      :", sum(svm_f1)/len(svm_f1))

RF Accuracy : 0.5350993377483444
RF F1       : 0.6392552746163019

XGB Accuracy: 0.5127483443708609
XGB F1      : 0.5746595957725993

SVM Accuracy: 0.5172185430463576
SVM F1      : 0.5584778799014398


In [104]:
print(fdf["Target"].value_counts(normalize=True))

Target
1    0.550407
0    0.449593
Name: proportion, dtype: float64


adding lag features cos i am not getting improvements from current setup

In [109]:
fdf["Return_Lag1"] = (
    fdf.groupby("Company")["Return"]
       .shift(1)
)

fdf["Return_Lag2"] = (
    fdf.groupby("Company")["Return"]
       .shift(2)
)

fdf["RSI_Lag1"] = (
    fdf.groupby("Company")["RSI"]
       .shift(1)
)

fdf["Momentum_Lag1"] = (
    fdf.groupby("Company")["Momentum"]
       .shift(1)
)

In [110]:
fdf.dropna(inplace=True)
fdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7245 entries, 2 to 7250
Data columns (total 24 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           7245 non-null   datetime64[ns]
 1   Open           7245 non-null   float64       
 2   High           7245 non-null   float64       
 3   Low            7245 non-null   float64       
 4   Close          7245 non-null   float64       
 5   Volume         7245 non-null   int64         
 6   Company        7245 non-null   object        
 7   Return         7245 non-null   float64       
 8   MA10           7245 non-null   float64       
 9   MA50           7245 non-null   float64       
 10  Volatility     7245 non-null   float64       
 11  Momentum       7245 non-null   float64       
 12  RSI            7245 non-null   float64       
 13  Range          7245 non-null   float64       
 14  MA_Ratio       7245 non-null   float64       
 15  Month          7245 non-nu

In [116]:
features_relative1= [
    "Return",
    "Return_Lag1",
    "Return_Lag2",
    "Momentum",
    "Momentum_Lag1",
    "RSI",
    "RSI_Lag1",
    "Volatility",
    "MA_Ratio",
    "Range",
    "Company_Code"
]

In [117]:
X = fdf[features_relative1]

y = fdf["Target"]

In [118]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

In [119]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

rf_acc = []
rf_f1 = []

for train_idx, test_idx in tscv.split(X):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        random_state=42
    )

    rf.fit(X_train, y_train)

    pred = rf.predict(X_test)

    rf_acc.append(
        accuracy_score(y_test, pred)
    )

    rf_f1.append(
        f1_score(y_test, pred)
    )

print("RF Accuracy:", sum(rf_acc)/len(rf_acc))
print("RF F1      :", sum(rf_f1)/len(rf_f1))

RF Accuracy: 0.535542667771334
RF F1      : 0.6440169195823097


In [120]:
rf.fit(X, y)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

print(
    importance.sort_values(
        "Importance",
        ascending=False
    )
)

          Feature  Importance
8        MA_Ratio    0.124519
7      Volatility    0.117276
2     Return_Lag2    0.104748
6        RSI_Lag1    0.103767
5             RSI    0.093618
9           Range    0.092513
4   Momentum_Lag1    0.089131
3        Momentum    0.086111
1     Return_Lag1    0.085677
0          Return    0.082378
10   Company_Code    0.020263


# new experiments on a new dataset